In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# load the dataset
df = pd.read_csv("/content/drive/MyDrive/Machine Learning/text_example.csv")

# drop rows where there's no text or no teacher label
df_clean = df.dropna(subset=['Text', 'Teacher'])

# merge teacher names that are the same person
df_clean['Teacher'] = df_clean['Teacher'].replace({
    "Dr. Rhyner Period 2": "Dr. Rhyner",
    "Dr.Morse": "Dr. Morse",
    "Ms Anderson": "Ms. Anderson",
    "Ms.Anderson": "Ms. Anderson",
    "Kira Morgan": "Ms. Morgan"
})

# function to clean up the text (make lowercase, remove punctuation)
def simple_preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

# apply the text cleaning to each row
df_clean['cleaned_text'] = df_clean['Text'].apply(simple_preprocess)

# count how many times each teacher appears
label_counts = df_clean['Teacher'].value_counts()

# only keep teachers that show up more than once
valid_labels = label_counts[label_counts > 1].index
df_filtered = df_clean[df_clean['Teacher'].isin(valid_labels)]

# convert text to TF-IDF features (using 1-word and 2-word phrases)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X = vectorizer.fit_transform(df_filtered['cleaned_text'])
y = df_filtered['Teacher']

# split into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# make the classifier and train it
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# run predictions on the test set
y_pred = clf.predict(X_test)

# show accuracy and more details
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

# get the list of all the feature names (words/phrases)
feature_names = vectorizer.get_feature_names_out()
X_train_array = X_train.toarray()

# figure out which words were most important for each teacher
print("\nTop TF-IDF features per teacher (based on training data):")
for i, label in enumerate(clf.classes_):
    class_indices = [j for j, val in enumerate(y_train) if val == label]
    mean_tfidf = np.mean(X_train_array[class_indices], axis=0)
    top_indices = mean_tfidf.argsort()[-10:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    print(f"\n{label}: {', '.join(top_features)}")


<ipython-input-16-3360e1af5a12>:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Teacher'] = df_clean['Teacher'].replace({
<ipython-input-16-3360e1af5a12>:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['cleaned_text'] = df_clean['Text'].apply(simple_preprocess)


Accuracy: 0.38461538461538464

Classification Report:

              precision    recall  f1-score   support

  Dr. Bufkin       0.00      0.00      0.00         1
  Dr. Chuang       0.25      0.50      0.33         2
   Dr. Ermer       0.00      0.00      0.00         1
 Dr. Margini       0.50      1.00      0.67         2
   Dr. Morse       0.00      0.00      0.00         1
  Dr. Rhyner       0.00      0.00      0.00         1
 Mr. McCrink       0.00      0.00      0.00         1
Ms. Anderson       0.40      1.00      0.57         2
 Ms. Cornick       0.00      0.00      0.00         1
  Ms. Morgan       0.00      0.00      0.00         1

    accuracy                           0.38        13
   macro avg       0.11      0.25      0.16        13
weighted avg       0.18      0.38      0.24        13


Top TF-IDF features per teacher (based on training data):

Dr. Brown: the, to, and, his, equiano, he, of, in, neo, the matrix

Dr. Bufkin: the, of, to, and, croesus, matrix, the matrix,

In [18]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# get a list of all unique teacher names
teachers = sorted(df_clean['Teacher'].unique())

# print them out so we know what to pick from
print("Available teachers:")
for t in teachers:
    print("-", t)

# ask the user which teacher they want to generate a word cloud for
selected_teacher = input("\nEnter the name of a teacher from the list above: ").strip()

# if the input is not valid, show an error
if selected_teacher not in teachers:
    print(f"\nTeacher '{selected_teacher}' not found. Please make sure the name is typed exactly as shown.")
else:
    # grab all the text for that teacher and make it one big string
    all_text = " ".join(df_clean[df_clean['Teacher'] == selected_teacher]['cleaned_text'])

    # generate the word cloud using that text
    wordcloud = WordCloud(width=1000, height=500, background_color='white', colormap='plasma').generate(all_text)

    # show the word cloud
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Word Cloud for {selected_teacher}", fontsize=20)
    plt.show()


Available teachers:
- Dr. Brown
- Dr. Bufkin
- Dr. Chuang
- Dr. Ermer
- Dr. Margini
- Dr. Morse
- Dr. Rhyner
- Macbeth Fall Essay
- Mr. Lefevere
- Mr. McCrink
- Mrs. Carson
- Mrs. Mandel
- Ms. Anderson
- Ms. Bustamante
- Ms. Cornick
- Ms. Morgan
- Ms. Morse
- Ms. Nero
- Ms.Berler
- World Literature Period 2

Enter the name of a teacher from the list above: hi

Teacher 'hi' not found. Please make sure the name is typed exactly as shown.
